# 🦩 Using Ibis-framework in Titanic 🦩

この Notebook では Kaggle の Titanic コンペを題材に Ibis のメソッドを学びます。

こちらの [Notebook](https://www.kaggle.com/sishihara/upura-kaggle-tutorial-01-first-submission) の各処理を Ibis で行っています。

# What`s Ibis ?
  
Ibis は統合的にデータ処理を実行可能なインターフェースを提供するライブラリで、 現在サポートしている18を超えるデータ処理ライブラリを同一の記法で使用することができます。利用ケースに応じてバックエンドのエンジンを柔軟に切り替えることができ、コードの書き換えコストを限りなく少なくすることができます。

### 2024年1月現在サポートしているエンジン
BigQuery , ClickHouse , Dask , DataFusion , Druid ,
DuckDB（ MotherDuck への接続もサポート） , Exasol ,
Flink , Impala , MSSQL , MySQL , Oracle , pandas ( CuDF ) , Polars ,
PostresSQL , PySpark , Snowflake , SQLite , Trino

- Ibis 100 本ノック  
https://github.com/kunishou/Ibis_100_knocks

- 公式 Reference  
https://ibis-project.org/  
  
- Github  
https://github.com/ibis-project  

# さぁ、Ibis メソッドを見ていきましょう！

# 事前準備

In [ ]:
# Ibis のインストール

!pip install /kaggle/input/ibis-framework/*.whl -qq  2>/dev/null

apache-beam, google-cloud-bigquery 等、いくつか使用しないライブラリで  
エラーが出ますが無視して大丈夫です。  
自分の notebook 上で ibis を使用したい場合は「Add data」で「ibis-framework」で検索すれば  
私のほうで登録した ibis のインストールに必要な whl ファイルデータセットが出てきます。  

In [ ]:
# ライブラリのインストール

import numpy as np
import pandas as pd
import ibis

In [ ]:
# インタラクティブモード（逐次評価）をオン
# EDA など各セルの結果を確認しながら進めたい場合は
# オンにする（遅延評価の場合は False を指定）

ibis.options.interactive = True

In [ ]:
# バックエンドを Polars に設定
# Ibis が対応しているフレームワークであれば切り替え可能
# （GPU 環境であれば cuDF も利用できます）

ibis.set_backend("polars")
# ibis.set_backend("pandas")
# ibis.set_backend("duckdb")

# データの読み込み

In [ ]:
# 読み込みファイルの確認

!ls ../input/titanic

In [ ]:
# csv ファイルの読み込み

train = ibis.read_csv("../input/titanic/train.csv")
test = ibis.read_csv("../input/titanic/test.csv")
gender_submission = ibis.read_csv("../input/titanic/gender_submission.csv")

csv ファイルの読み込みは .read_csv() を使用する。  
引数はバックエンドに指定するフレームワークにより異なる。  
例えば、 pandas バックエンドの場合は skiprows などを指定可能。  
（ただし、pandas.read_csv のすべての引数を使えるわけではない）  

In [ ]:
# テーブルの先頭 5 行までを表示

gender_submission.head()

ibis での表示結果は データフレームではなく  
データベーステーブルというオブジェクトになる。  
.head() でテーブルの先頭 5行 までを表示する。  

In [ ]:
# pandas データフレームでの表示

gender_submission.execute().head()

.execute() メソッドを使うことで pandas データフレームに変換される。  
pandas データフレームの状態であれば pandas メソッドを適用できる。  
なお、遅延評価モードの際にコードの最後に .execute() を付けることで  
コードを実行することが可能です。  

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
# train と test を行方向に結合

data = ibis.union(train, test.mutate(ibis.literal(-1).cast("int64").name("Survived")))
data.head()

テーブル同士の行方向の結合には .union() を使用する。  
.union() では列数やデータ型など2つのテーブルの構造が  
異なると結合できない点に注意。  
上記では test テーブルにダミーで Survived 列を作成して  
結合している。  

In [ ]:
# 各テーブルの行数を表示

print(train.count(), test.count(), data.count())

In [ ]:
# 欠損値数を確認

data.agg(
    data[col].isnull().sum().name(col) for col in data.columns
)

合計、平均、最大値、最小値などのデータ集約関連操作をする場合は  
.aggregate() もしくは .agg() メソッドを使用する。  
（以下の例では Survived 列の欠損値数が 0 だが test テーブルには  
 Survived 列は存在せず本来は欠損している点に注意）  

# データ前処理

## 1. Sex

In [ ]:
# "male" を 0 に、 "female" を 1 に置換する。

data2 = data.mutate(
    data["Sex"].case().when("male", 0)
                      .when("female", 1)
                      .end()
                      .cast("int64")
                      .name("Sex")
)
data2.head()

値の置換には mutate() と case().when()._else().end() 構文を使用する。  
mutate 内で記述した操作を .name() で指定した列に適用する。  
.name() で新規の列を指定すれば新しい列を追加できる。  
.name() を使わない方法もある。  

## 2. Embarked

In [ ]:
# 欠損値を "S" で補完し、その後に、"S" を 0 、"C" を 1 、"Q" を 2 に
# 置換し、データ型を "int64" に変換

data2 = data.mutate(
    Embarked=data["Embarked"].fillna("S")
).mutate(
    Embarked=data["Embarked"].case().when("S", 0)
                             .when("C", 1)
                             .when("Q", 2)
                             .end()
                             .cast("int64")
)
data2.head()

mutate内で 「 Embarked= 」という形で操作を適用する  
列を指定することも可能。.name() だと mutate 内での  
操作は 1 操作のみだがこの記法で , 区切りで続けて  
操作を記述することができる。  
欠損値補完は .fillna() を使用する。  
データ型の変換は .cast() を使用する。

## 3. Fare

In [ ]:
# 欠損値を平均値で補完

data2 = data.mutate(
    Fare=data["Fare"].fillna(data["Fare"].mean().execute())
)
data2.head()

fillna(data["Fare"].mean().execute()) というように .execute() で平均値集計を  
実行しておく（遅延評価モードでは不要）

## 4. Age

In [ ]:
# 欠損値を age_avg - age_std と age_avg + age_std の間のランダムな整数で補完

age_avg = data['Age'].mean().execute()
age_std = data['Age'].std().execute()

data2 = data.mutate(
    Age=data["Age"].fillna(np.random.randint(age_avg - age_std, age_avg + age_std))
)

data2.head()

# データ前処理の一括実施
  
Ibis では処理内容を数珠つなぎにまとめることができコードの高い可読性を実現できます。

In [ ]:
age_avg = data['Age'].mean().execute()
age_std = data['Age'].std().execute()

# データ前処理
data2 = data.mutate(
        Sex=data["Sex"].case().when("male", 0)
                          .when("female", 1)
                          .end()
                          .cast("int64"),
        Embarked=data["Embarked"].fillna("S").case().when("S", 0)
                                             .when("C", 1)
                                             .when("Q", 2)
                                             .end()
                                             .cast("int64"),
        Fare=data["Fare"].fillna(data["Fare"].mean().execute()),
        Age=data["Age"].fillna(np.random.randint(age_avg - age_std, age_avg + age_std))
    )

data2.head()

In [ ]:
# 必要列の選択

select_columns = ['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked']
data3 = data2.select(select_columns)
data3.head()

もしくは不要列の削除でも良い。

In [ ]:
# 不要列の削除

delete_columns = ['Name', 'PassengerId', 'SibSp', 'Parch', 'Ticket', 'Cabin']
data3 = data2.drop(delete_columns)
data3.head()

In [ ]:
train = data3[0:int(train.count().execute())]
test = data3[int(train.count().execute()):]

In [ ]:
# データフレーム形式に変換
# これ以降は pandas などと同じ操作

y_train = train["Survived"].execute()
X_train = train.drop("Survived").execute()
X_test = test.drop("Survived").execute()

In [ ]:
X_train.head()

In [ ]:
y_train.head()

# 機械学習アルゴリズム

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
clf = LogisticRegression(penalty='l2', solver="sag", random_state=0)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
y_pred[:20]

# 提出

In [ ]:
sub = gender_submission.execute()
sub['Survived'] = list(map(int, y_pred))
sub.to_csv("submission.csv", index=False)
sub.head()